# Read the Combine through body size: three positions, three different stories
*NFL Big Data Bowl 2027 · Open Track · Combine tracking linked to regular-season game tracking · WR, TE, DB, DL*

**Read Combine numbers through body size, and three positions tell three different stories.** Receivers: speed is real and shows up on Sunday. Linemen: the drills predict pass rush, mostly through body size, so read them size-adjusted. Defensive backs: the Combine alone is a weak guide at this sample size.

**The required link, stated plainly.** We connect each rookie's 10 Hz Combine tracking (2023 to 2025 classes) to his regular-season game tracking. A receiver's tracked Combine top speed predicts his regular-season speed on deep routes (ρ +0.63 year-controlled, **+0.58 with body size held fixed**, n 62). The cell below recomputes that from the raw tracking files.

Data: the competition's Combine tracking, game tracking, play, player and game files, plus free public Pro-Football-Reference stats (two DB outcomes). No draft-position information is used; draft *year* is a control only. Running backs are excluded. **Sunday Speed** is a player's 90th-percentile top speed per play where his job asks for speed (deep routes for receivers; man or deep coverage for defensive backs), regular season. **Burst Score** is a 0 to 100 within-position blend of the tracked 20-yard time and the jumps. ρ is a rank correlation; positive means the better athlete does better.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.lines as ml
from scipy import stats
from scipy.stats import norm
from IPython.display import Image, display

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
OKABE = {"blue": "#0072B2", "orange": "#E69F00", "red": "#D55E00", "green": "#009E73", "grey": "#8a8a8a"}
ND = "NOT DETECTED (low power)"

def find_out():
    for p in [Path("out"), Path("../out"), Path("/kaggle/working/out")]:
        if (p / "scorecard_full.csv").exists():
            return p
    hits = list(Path("/kaggle/input").rglob("scorecard_full.csv")) if Path("/kaggle/input").exists() else []
    if hits:
        return hits[0].parent
    raise FileNotFoundError("Attach the 'bdb2027-outputs' dataset (it holds the out/ folder).")

def find_comp():
    for p in [Path("data"), Path("../data")]:
        if (p / "combine_tracking.csv").exists():
            return p
    hits = list(Path("/kaggle/input").rglob("combine_tracking.csv")) if Path("/kaggle/input").exists() else []
    return hits[0].parent if hits else None

OUT, COMP = find_out(), find_comp()
FIGS = OUT / "figs"
SAVE = Path("/kaggle/working/writeup_figs") if Path("/kaggle/working").exists() else Path("writeup_figs")
SAVE.mkdir(parents=True, exist_ok=True)
full = pd.read_csv(OUT / "scorecard_full.csv")
card = pd.read_csv(OUT / "scorecard.csv")
wt = pd.read_csv(OUT / "scorecard_weight.csv")
g25 = full[full.n >= 25]
qtxt = lambda q: "<0.01" if q < .01 else f"{q:.2f}"
print("audited outputs:", OUT, "| competition data:", COMP)

In [ ]:
# shared style: one verdict palette (Okabe-Ito, colour-blind safe) used by every figure; readable at Kaggle width (about 8.5 in wide, 9 pt+)
VCOL = {"TRANSLATES": "#009E73", "LEANS": "#E69F00", "SIZE, NOT SPEED": "#CC79A7", ND: "#7f7f7f", "TOO FEW TO TELL": "#bdbdbd"}
plt.rcParams.update({"font.size": 9.5, "axes.titlesize": 10.5, "axes.labelsize": 9.5, "xtick.labelsize": 9, "ytick.labelsize": 9, "figure.dpi": 110, "savefig.dpi": 150})

def draw_ci(ax, y, lo, hi, n, col, lw=2.4, alpha=1.0):
    """95% interval; hidden below 15 players (the bootstrap is unreliable there) and flagged on the plot."""
    if n < 15 or not np.isfinite(lo) or not np.isfinite(hi):
        ax.text(0.02, y, "interval not shown (n<15)", fontsize=7.5, color=col, va="center"); return
    ax.plot([lo, hi], [y, y], color=col, lw=lw, alpha=alpha, solid_capstyle="round")

def tag(n, q, extra=""):
    return f"n {int(n)} · q {qtxt(q)}" + (f" · {extra}" if extra else "")

import textwrap
def finish(fig, name, title, foot="", legend=None, rect_lr=(0.0, 1.0), top_pad=0.0):
    """Left-aligned takeaway title, optional legend row under it, wrapped footnote at the bottom, then save. Keeps text inside the canvas."""
    W, H = fig.get_size_inches(); line = 0.135 / H
    lines = textwrap.wrap(foot, int(W * 14.5)) if foot else []
    if lines: fig.text(.01, .006, "\n".join(lines), fontsize=7.5, color="#444", va="bottom")
    fig.suptitle(title, x=.01, y=1 - .06 / H, ha="left", va="top", fontweight="bold", fontsize=10.5)
    top = 1 - (.40 if legend else .30) / H - top_pad
    if legend: fig.legend(legend[0], legend[1], loc="upper left", bbox_to_anchor=(0.01, 1 - .30 / H), ncol=legend[2] if len(legend) > 2 else 3, fontsize=8.5, frameon=False, handletextpad=.4, columnspacing=1.2)
    fig.tight_layout(rect=(rect_lr[0], .006 + line * len(lines) + .01, rect_lr[1], top)); fig.savefig(SAVE / name); plt.show()

In [ ]:
# ---- THE GATE: Combine tracking -> regular-season game tracking, receivers, recomputed from the raw competition files ----
MPH = 2.0455  # yards/second to mph
gate = full.query("group == 'WR' and test == 'c_top_mph' and outcome == 'sunday_speed_mph'").iloc[0]
if COMP is not None:
    pl = pd.read_csv(COMP / "players.csv", usecols=["nfl_id", "nfl_position", "draft_year"])
    wr = pl[pl.nfl_position == "WR"]; ids = set(wr.nfl_id)
    # Combine side: fastest speed in any 40-yard-dash attempt, from the 10 Hz sensor
    ct = pd.read_csv(COMP / "combine_tracking.csv", usecols=["nfl_id", "entity_type", "drill_type", "s"])
    ct = ct[(ct.entity_type == "PLAYER") & (ct.drill_type == "FORTY_YARD_DASH") & ct.nfl_id.isin(ids)]
    top = (ct.groupby("nfl_id").s.max() * MPH).rename("combine_top_mph")
    # Game side: regular-season deep routes (go, post, corner, wheel) that stood; top speed on each play from game tracking
    games = pd.read_csv(COMP / "games.csv", usecols=["game_id", "season_type"])
    pp = pd.read_csv(COMP / "player_play.csv", usecols=["game_id", "play_id", "nfl_id", "route_ran", "play_nullified_by_penalty"])
    pp = pp[pp.nfl_id.isin(ids) & pp.route_ran.isin(["GO", "POST", "CORNER", "WHEEL"]) & (pp.play_nullified_by_penalty == "N")
            & pp.game_id.isin(games.game_id[games.season_type == "REG"])]
    parts = []
    for yr in (2023, 2024, 2025):
        for ch in pd.read_csv(COMP / f"game_tracking_{yr}.csv", usecols=["game_id", "play_id", "nfl_id", "s"], dtype={"s": "float32"}, chunksize=2_000_000):
            ch = ch[ch.nfl_id.isin(ids)]
            if len(ch):
                parts.append(ch.groupby(["game_id", "play_id", "nfl_id"], sort=False).s.max().reset_index())
    plays = pd.concat(parts).groupby(["game_id", "play_id", "nfl_id"], as_index=False).s.max()
    plays = plays.merge(pp[["game_id", "play_id", "nfl_id"]], on=["game_id", "play_id", "nfl_id"])
    cnt = plays.groupby("nfl_id").s.size()
    sunday = (plays.groupby("nfl_id").s.quantile(0.90) * MPH)[cnt >= 20].rename("sunday_speed_mph")   # at least 20 deep routes
    cr = pd.read_csv(COMP / "combine_results.csv", usecols=["nfl_id", "combine_weight", "combine_height"]).drop_duplicates("nfl_id").set_index("nfl_id")
    d = pd.concat([top, sunday], axis=1, join="inner").join(wr.set_index("nfl_id").draft_year).join(cr).dropna()
    x, y = d.combine_top_mph, d.sunday_speed_mph
    rho_year = stats.spearmanr(x - x.groupby(d.draft_year).transform("mean"), y - y.groupby(d.draft_year).transform("mean"))[0]
    # rank partial correlation holding draft year, weight and height fixed (same recipe as the audited fit): rank, residualise, correlate
    Z = np.column_stack([np.ones(len(d)), pd.get_dummies(d.draft_year.astype(str), drop_first=True).to_numpy(float), stats.rankdata(d.combine_weight), stats.rankdata(d.combine_height)])
    res = lambda v: stats.rankdata(v) - Z @ np.linalg.lstsq(Z, stats.rankdata(v), rcond=None)[0]
    rx, ry = res(x), res(y); rho_size = np.corrcoef(rx, ry)[0, 1]
    rng = np.random.default_rng(7); xs = np.linspace(x.min(), x.max(), 60); bands, slopes = [], []
    for _ in range(1500):
        i = rng.integers(0, len(d), len(d)); b = np.polyfit(x.to_numpy()[i], y.to_numpy()[i], 1); slopes.append(b[0]); bands.append(np.polyval(b, xs))
    lo_b, hi_b = np.percentile(bands, [2.5, 97.5], axis=0); s_lo, s_hi = np.percentile(slopes, [2.5, 97.5]); slope = np.polyfit(x, y, 1)
    print(f"Receivers with a tracked 40, size data and 20+ regular-season deep routes: n={len(d)} (audited n={int(gate.n)})")
    print(f"Spearman, draft year held fixed: {rho_year:+.2f} (audited {gate.rho_raw:+.2f}); rank partial with size held fixed: {rho_size:+.2f} (audited {gate.rho_size:+.2f}); slope {slope[0]:.2f} mph per mph (bootstrap 95% CI {s_lo:.2f} to {s_hi:.2f})")
    assert abs(len(d) - int(gate.n)) <= 1 and abs(rho_size - gate.rho_size) < 0.05, "live recomputation disagrees with the audited table"
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 4.3))
    c = VCOL["TRANSLATES"]
    a.fill_between(xs, lo_b, hi_b, color=c, alpha=.18, lw=0); a.plot(xs, np.polyval(slope, xs), color=c, lw=1.8)
    a.scatter(x, y, s=20, color=c, alpha=.75, edgecolor="white", linewidth=.4)
    a.set_xlabel("Combine top speed in the 40 (mph)"); a.set_ylabel("Sunday Speed on deep routes (mph)")
    a.set_title("A. As measured", loc="left", fontweight="bold")
    a.text(.03, .97, f"n {len(d)}; Spearman {rho_year:+.2f} (year fixed)\n{slope[0]:+.2f} mph per mph\n(line and 95% band: straight-line fit)", transform=a.transAxes, va="top", fontsize=8.5)
    b.scatter(rx, ry, s=20, color=c, alpha=.75, edgecolor="white", linewidth=.4); b.axhline(0, color="k", lw=.6); b.axvline(0, color="k", lw=.6)
    bb = np.polyfit(rx, ry, 1); xx = np.linspace(rx.min(), rx.max(), 2); b.plot(xx, np.polyval(bb, xx), color=c, lw=1.8)
    b.set_xlabel("Combine top speed, rank left over\nafter year, weight, height"); b.set_ylabel("Sunday Speed, rank left over")
    b.set_title("B. With body size held fixed", loc="left", fontweight="bold")
    b.text(.03, .97, f"partial rank ρ {gate.rho_size:+.2f}\n95% CI {gate.ci_lo:+.2f} to {gate.ci_hi:+.2f}\nq {gate.fdr_q:.3f}", transform=b.transAxes, va="top", fontsize=8.5)
    fig.tight_layout(); fig.savefig(SAVE / "fig1_gate_wr.png"); plt.show()
else:
    print("Competition data not attached: showing the saved gate figure and audited numbers instead.")
    display(Image(str(FIGS / "fig1_gate_wr.png")))
ok = (gate.verdict == "TRANSLATES") and (gate.rho_raw > 0)
print(f"Audited: size-controlled rho {gate.rho_size:+.2f} (95% CI {gate.ci_lo:+.2f} to {gate.ci_hi:+.2f}), q {gate.fdr_q:.3f}, better on held-out draft classes in {gate.classes_helped}")
print("GATE (Combine tracking -> regular-season performance):", "PASS" if ok else "FAIL")

## 1. Receivers: speed is real, and it carries to Sunday
Two Combine measures clear every check, and both are receiver straight-line speed. The 40-yard dash goes with faster game speed on deep routes (ρ +0.52 with size held fixed, 95% CI +0.30 to +0.68, n 63, q <0.01). The tracked Combine top speed does so even more strongly (+0.58, +0.37 to +0.74, n 62, q <0.01). Both improve predictions on held-out draft classes (3/3 each).

Each extra mph of Combine top speed goes with about 0.67 mph more on a deep route in a game (Figure 1). Size barely moves this result (+0.63 before size control, +0.58 after), unlike the defensive line. The other receiver tests (10-yard split, jumps, shuttle, Burst Score) were not detected against deep-route speed (about 0.37 or more was needed to see them).

Figure 2 is the proof of method. Of 164 tests with at least 25 players, 28 reach p < 0.05 against about 8 expected by chance (cells share players, so that overstates independent findings). After false-discovery correction exactly 2 remain, both receiver rows. The method finds real signal where it exists.

*Figure 1. Each dot is a receiver. A: as measured. B: after removing draft year, weight and height from both axes (ranks).*

In [ ]:
tl = [("forty", "40-yard dash"), ("c_top_mph", "Top speed (tracked)"), ("ten_yd_split", "10-yard split"), ("burst_0_100", "Burst Score †"), ("vertical", "Vertical jump"), ("broad_jump", "Broad jump"), ("short_shuttle", "Short shuttle"), ("three_cone", "3-cone")]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8.6, 8.4), gridspec_kw={"height_ratios": [1.15, 1]})
for i, (t, lab) in enumerate(tl[::-1]):
    r = full[(full.group == "WR") & (full.subgroup == "All") & (full.test == t) & (full.outcome == "sunday_speed_mph")].iloc[0]; col = VCOL[r.verdict]
    draw_ci(a1, i, r.ci_lo, r.ci_hi, r.n, col, 2.4); a1.scatter(r.rho_size, i, s=52, color=col, zorder=3); a1.scatter(r.rho_raw, i, s=40, facecolor="white", edgecolor=col, marker="D", zorder=3)
    a1.text(1.01, i, tag(r.n, r.fdr_q, f"held-out {r.classes_helped}"), va="center", fontsize=8, color="#333", transform=a1.get_yaxis_transform())
a1.set_yticks(range(len(tl))); a1.set_yticklabels([x[1] for x in tl[::-1]]); a1.axvline(0, color="k", lw=.8); a1.set_xlim(-.9, .9)
a1.set_title("A. Game speed on deep routes: only straight-line speed stands out", loc="left", fontweight="bold", fontsize=9.5)
a1.set_xlabel("rank correlation; + = better athlete, faster Sunday speed\nfilled dot = size held fixed with 95% CI; hollow diamond = before size control")
p = g25.sort_values("p_perm").reset_index(drop=True); k = np.arange(1, len(p) + 1); pass_ = p.fdr_q < .05
a2.scatter(k[~pass_], p.p_perm[~pass_], s=16, color=VCOL[ND], alpha=.8, label="does not clear q < 0.05"); a2.scatter(k[pass_], p.p_perm[pass_], s=64, color=VCOL["TRANSLATES"], zorder=3, label="clears q < 0.05 (the two receiver speed rows)")
a2.plot(k, k / len(p), color="k", ls=":", lw=1.2, label="expected if no test had any real link"); a2.axhline(.05, color="#444", ls="--", lw=1, label="p = 0.05")
a2.set_yscale("log"); a2.set_ylim(5e-5, 1.3); a2.set_xlabel(f"the {len(p)} tests with at least 25 players, ranked by p-value"); a2.set_ylabel("permutation p-value (log scale)\nfloor 0.0001 = 10,000 shuffles")
a2.set_title(f"B. {int((p.p_perm < .05).sum())} tests under p = 0.05; about {round(len(p) * .05)} expected by chance; {int(pass_.sum())} survive correction", loc="left", fontweight="bold"); a2.legend(fontsize=8, frameon=False, loc="lower right")
finish(fig, "fig2_wr_control.png", "Receivers: one speed signal clears every check, out of 164 tests", "† Burst Score recipe was tuned using game results (optimistic). q is the Benjamini-Hochberg rate within each family of tests; tests share players, so they are not independent.", rect_lr=(0, .86))

*Figure 2. A: every receiver test against deep-route speed. B: all 164 tests; only two survive correction.*

## 2. Defensive line: the drills predict pass rush, mostly through body size
Linemen's Combine numbers do carry information: faster linemen get off quicker, rush more and sack more. The catch is how much is body size. Across the four outcomes with a strong raw link (get-off, pressure, sack, tackle for loss), 32 test-by-outcome cells have at least 25 players. Their median year-controlled link is ρ +0.68. Hold weight, height and role (edge or interior) fixed and the median is +0.21, so on the median about 69% of the raw link is shared with body size and role (a statistical split, not proof of cause; Figure 3).

What remains is still positive and still useful:
- The 40 against **pressure rate** goes from +0.71 to +0.35 (95% CI +0.11 to +0.54, q 0.16).
- The 10-yard split against **get-off speed** goes from +0.80 to +0.32 (q 0.24).
- The 40 against **sack rate** goes from +0.72 to +0.23, and its interval (-0.04 to +0.48) now touches zero.

**How scouts should read it:** a lineman's 40 is a size-adjusted signal. Compare him with linemen of the same weight and role, not the whole group. The pressure lean appears separately for edge (+0.35, n 39) and interior (+0.40, n 33) rushers. Of 33 defensive-line cells with a raw link of at least 0.30, 32 lose half or more to size; none of 6 receiver or DB cells do.

In [ ]:
tests = [("ten_yd_split", "10-yard split"), ("forty", "40-yard dash"), ("c_top_mph", "Top speed (tracked)"), ("burst_0_100", "Burst Score †"), ("vertical", "Vertical jump"), ("broad_jump", "Broad jump")]
outs = [("speed_1s_mph", "Get-off speed (1 s)"), ("pressure_rate", "Pressure rate"), ("sack_rate", "Sack rate"), ("tfl_rate", "Tackles for loss")]
fig, axes = plt.subplots(2, 2, figsize=(8.6, 7.4), sharex=True)
used = []
for ax, (oc, ol) in zip(axes.ravel(), outs):
    for i, (t, tl_) in enumerate(tests[::-1]):
        r = g25[(g25.group == "DL") & (g25.subgroup == "All") & (g25.test == t) & (g25.outcome == oc)]
        if r.empty: continue
        r = r.iloc[0]; col = VCOL[r.verdict]; used.append(r)
        ax.annotate("", xy=(r.rho_size, i), xytext=(r.rho_raw, i), arrowprops=dict(arrowstyle="->", color=col, lw=1.2, alpha=.6))
        draw_ci(ax, i, r.ci_lo, r.ci_hi, r.n, col, 2.6, .55)
        ax.scatter(r.rho_raw, i, s=40, facecolor="white", edgecolor=col, marker="D", zorder=3); ax.scatter(r.rho_size, i, s=50, color=col, zorder=4)
    ax.set_yticks(range(len(tests))); ax.set_yticklabels([t[1] for t in tests[::-1]]); ax.axvline(0, color="k", lw=.8); ax.set_xlim(-.45, .95); ax.set_title(ol, fontweight="bold", loc="left")
for ax in axes[1]: ax.set_xlabel("rank correlation (+ = better athlete)")
U = pd.DataFrame(used); QMIN, NMIN, NMAX = U.fdr_q.min(), int(U.n.min()), int(U.n.max())
finish(fig, "fig3_dl_size.png", "Defensive line: most of the raw link shrinks once body size and role are held fixed",
       f"n {NMIN} to {NMAX} per cell; none clears q < 0.05 (lowest q {QMIN:.2f}). Raw links carry no interval. 3-cone and shuttle (under 36 linemen) omitted. † Burst Score recipe tuned using game results.",
       legend=([ml.Line2D([], [], marker="D", color="w", markerfacecolor="w", markeredgecolor="#333", lw=0), ml.Line2D([], [], marker="o", color="#333", lw=0), ml.Line2D([], [], marker="o", color=VCOL["SIZE, NOT SPEED"], lw=0), ml.Line2D([], [], marker="o", color=VCOL["LEANS"], lw=0)],
               ["raw link (year fixed)", "adjusted, 95% interval", "mostly size", "lead (q not cleared)"], 4))

*Figure 3. Defensive-line tests, raw to size-adjusted. The gap is what body size and role account for.*

## 3. Consistent leads that need more seasons
Five results hold in every held-out draft class (3/3 each) but do not clear the false-discovery bar. We call them leads and show the honest q:
- **Receivers:** a higher vertical jump (+0.34, n 58, q 0.18) and Burst Score (+0.31, n 54, q 0.18) go with more separation on break routes.
- **Defensive line:** the 40 goes with pressure rate (+0.35, q 0.16), the 10-yard split with get-off speed (+0.32, q 0.24), and Burst Score with tackles for loss (+0.32, q 0.44).

Each estimate is at or below what these samples reliably detect even before correction (about 0.37 for receivers, 0.37 for linemen), and well below what the correction demands (about 0.51 and 0.56). If the true links equal today's estimates, confirming them needs roughly 4 to 9 more draft classes (Figure 4). That is optimistic: leads are the best-looking tests.

The textbook agility stories were not detected. We pre-declared that 3-cone and shuttle matter more on break than vertical routes (differences -0.24 and -0.11, intervals include zero) and that quickness matters more against man than zone (10-yard difference +0.15, CI -0.13 to +0.41). Only 22 to 30 receivers ran those drills, so only effects of about 0.57 or more were visible; not detected is not ruled out.

In [ ]:
def players_needed(rho, alpha, k=4):
    return int(np.ceil(1.06 * ((norm.ppf(1 - alpha / 2) + norm.ppf(.8)) / np.arctanh(rho)) ** 2 + k + 3))
LEADS = [("WR", "All", "vertical", "break_y", "Receivers: vertical jump → break separation"),
         ("WR", "All", "burst_0_100", "break_y", "Receivers: Burst Score † → break separation"),
         ("DL", "All", "forty", "pressure_rate", "Linemen: 40-yard dash → pressure rate"),
         ("DL", "All", "ten_yd_split", "speed_1s_mph", "Linemen: 10-yard split → get-off speed"),
         ("DL", "All", "burst_0_100", "tfl_rate", "Linemen: Burst Score † → tackles for loss")]
fig, ax = plt.subplots(figsize=(8.6, 4.6)); col = VCOL["LEANS"]
for i, (grp, sub, t, oc, lab) in enumerate(LEADS[::-1]):
    r = full[(full.group == grp) & (full.subgroup == sub) & (full.test == t) & (full.outcome == oc)].iloc[0]
    ax.fill_betweenx([i - .34, i + .34], -r.mde80, r.mde80, color="#d9d9d9", alpha=.7, zorder=0)
    draw_ci(ax, i, r.ci_lo, r.ci_hi, r.n, col, 2.8); ax.scatter(r.rho_size, i, s=60, color=col, zorder=3)
    ax.text(1.01, i, f"{tag(r.n, r.fdr_q)}\nheld-out {r.classes_helped}\n~{players_needed(r.rho_size, r.alpha_bh)} players to confirm", va="center", fontsize=8, color="#333", transform=ax.get_yaxis_transform())
ax.set_yticks(range(len(LEADS))); ax.set_yticklabels([x[4] for x in LEADS[::-1]]); ax.axvline(0, color="k", lw=.8); ax.set_xlim(-.55, .8)
ax.set_xlabel("size-controlled rank correlation; + = better athlete, better result\ndot = estimate, line = 95% interval, gray band = smallest link this n reliably sees")
finish(fig, "fig4_leads.png", "Five leads: consistent in held-out classes, not yet past the correction",
       "Leads are the best-looking of many tests, so true links are probably smaller. 'Players to confirm' assumes today's estimate is true (optimistic). † Burst Score recipe tuned using game results.", rect_lr=(0, .79))

*Figure 4. The five leads. Gray band: what each sample could reliably see. Right label: players needed to confirm the estimate after correction, if it is true.*

## 4. Defensive backs: the Combine alone is a weak guide at this sample size
48 cornerback and safety cells have at least 25 players. 45 are **not detected** and 3 lean; none clears the correction (lowest q 0.75). The Combine is not useless for defensive backs; with about 59 players, only links of ρ 0.38 or more would be seen 8 times in 10, and a true link of 0.30 only 57% of the time (7% after correction; Table 1).

The intervals are wide. The 40 against missed tackles is +0.16 (CI -0.11 to +0.40); DB-drill braking against tackle rate is -0.16 (-0.40 to +0.12). Absence of evidence is not evidence of absence.

One result runs the wrong way. A faster 40 goes with *more* yards allowed per target (-0.35, n 53, 95% CI -0.58 to -0.05), and broad jump agrees (-0.35). We read that as coverage role, not "speed hurts", and it does not clear correction. Use the Combine for defensive backs as one input beside tape and role. Table 1 shows what every group's sample can see; tight ends (about 20 players, 0.66 detectable) are too few to say anything.

In [ ]:
def power_table(full):
    g = full[full.n >= 25]
    spec = [("Receivers", g[g.group == "WR"]),
            ("Defensive line, all", g[(g.group == "DL") & (g.subgroup == "All")]),
            ("   edge only", g[(g.group == "DL") & (g.subgroup == "Edge")]),
            ("   interior only", g[(g.group == "DL") & (g.subgroup == "Interior")]),
            ("Defensive backs", g[g.group == "DB"]),
            ("Tight ends (n 15+)", full[(full.group == "TE") & (full.n >= 15)])]
    rows = [dict(Group=lab, **{"Typical n": int(d.n.median()), "Smallest |rho| seen 8 times in 10": round(float(d.mde80.median()), 2),
                               "...after multiple-test correction": round(float(d.mde80_bh.median()), 2),
                               "Chance to see a true rho of 0.30": float(d.power_rho03.median()),
                               "...after correction": float(d.power_rho03_bh.median())}) for lab, d in spec]
    return pd.DataFrame(rows)


pt = power_table(full)
tbl = pt.copy()
for c in ["Chance to see a true rho of 0.30", "...after correction"]:
    tbl[c] = (100 * tbl[c]).round(0).astype(int).astype(str) + "%"
display(tbl.reset_index(drop=True))

db = card[card.group == "DB"]
fig, ax = plt.subplots(figsize=(8.6, 5.0))
labs = []
for i, (_, r) in enumerate(db.iloc[::-1].iterrows()):
    col = VCOL[r.verdict]
    ax.fill_betweenx([i - .34, i + .34], -r.mde80, r.mde80, color="#d9d9d9", alpha=.7, zorder=0)
    draw_ci(ax, i, r.ci_lo, r.ci_hi, r.n, col, 2.8); ax.scatter(r.rho_size, i, s=56, color=col, zorder=3)
    ax.text(1.01, i, tag(r.n, r.fdr_q) + ("\nruns the wrong way" if (r.rho_size < 0 and r.verdict == "LEANS") else ""), va="center", fontsize=8, color="#333", transform=ax.get_yaxis_transform())
    labs.append(f"{r.test_label.replace(' (tracking)', '')} → {r.outcome_label.replace(' (Sunday Speed)', '')}")
ax.set_yticks(range(len(db))); ax.set_yticklabels(labs, fontsize=8.5)
ax.axvline(0, color="k", lw=.8); ax.set_xlim(-.7, .8)
ax.set_xlabel("size-controlled rank correlation; + = better athlete, better result\ndot = estimate, line = 95% interval, gray band = smallest link this n reliably sees")
finish(fig, "fig5_db_power.png", "Defensive backs: wide intervals, most inside what this n can see",
       "One test per result, the one with the smallest p, so these intervals flatter the Combine. Not detected is not ruled out.", rect_lr=(0, .85),
       legend=([ml.Line2D([], [], marker="o", color=VCOL[k], lw=0) for k in ["LEANS", ND]], ["lead (interval excludes 0, q not cleared)", "NOT DETECTED (low power)"], 2))

*Table 1. What each sample can see: the smallest correlation seen 8 times in 10, and the chance of seeing a true 0.30 (medians over tests with 25+ players).*

*Figure 5. Defensive backs, best test per result: mostly inside the gray band, so not detected, not ruled out.*

## 5. Weight is the variable scouts must adjust for
A stopwatch cannot tell weight from talent. Holding height and draft year fixed (role too for linemen; Figure 6):

- **Defensive line.** Heavier linemen get off the ball slower (ρ -0.49, 95% CI -0.62 to -0.30, q <0.01) and reach a lower speed one second after the snap (-0.53, q <0.01). This is why Section 2 reads linemen size-adjusted.
- **Receivers.** Heavier receivers run slower on deep routes (-0.43, q <0.01) and slower than their own 40 predicts (-0.38, q 0.04).
- **Separation.** Heavier receivers also get less separation on break (-0.25), vertical (-0.28) and crosser (-0.29) routes, not on screens (+0.06): a lead (q 0.07).

Weight does not move everything: pressure rate is only weakly tied to it (-0.20, q 0.64). Scouts adjust a 40 for weight by feel; these numbers say how much.

In [ ]:
rows = [("DL", "player_get_off", "Linemen: get-off time"), ("DL", "speed_1s_mph", "Linemen: speed 1 s after snap"), ("DL", "pen_pass_2s", "Linemen: pass-rush depth at 2 s"),
        ("DL", "pressure_rate", "Linemen: pressure rate"), ("DL", "sack_rate", "Linemen: sack rate"),
        ("WR", "sunday_speed_mph", "Receivers: deep-route game speed"), ("WR", "ssoe_mph", "Receivers: game speed vs own 40"), ("WR", "sep_all_mean", "Receivers: separation, all routes")]
fig, ax = plt.subplots(figsize=(8.6, 4.9))
for i, (grp, outc, lab) in enumerate(rows[::-1]):
    r = wt[(wt.group == grp) & (wt.outcome == outc)].iloc[0]; clear = r.fdr_q < .05
    col = VCOL["TRANSLATES"] if clear else VCOL[ND]
    draw_ci(ax, i, r.ci_lo, r.ci_hi, r.n, col, 2.6); ax.scatter(r.rho_height_ctrl, i, s=54, color=col, zorder=3)
    ax.scatter(r.rho_raw, i, s=40, facecolor="white", edgecolor=col, marker="D", zorder=3)
    ax.text(1.01, i, tag(r.n, r.fdr_q), va="center", fontsize=8, color="#333", transform=ax.get_yaxis_transform())
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[2] for r in rows[::-1]]); ax.axvline(0, color="k", lw=.8); ax.set_xlim(-1, 1)
ax.set_xlabel("rank correlation of weight with the result (negative: heavier does worse)\nfilled dot = height and draft year fixed (linemen: role too), 95% interval; hollow diamond = draft year only")
finish(fig, "fig6_weight.png", "Weight: large where it matters, absent where it does not", "",
       rect_lr=(0, .86), legend=([ml.Line2D([], [], marker="o", color=VCOL[k], lw=0) for k in ["TRANSLATES", ND]], ["clears q < 0.05", "does not clear q < 0.05"], 2))

*Figure 6. Weight against on-field results. Left of zero: heavier players do worse.*

## How a scout uses this
A scorecard, not a ranking (Figure 7), behind a lookup dashboard: enter a Combine line and each test returns a tag and a one-line read. One example:
- **A faster 40 shows up as faster game speed on deep routes. It holds after size control and on a held-out draft class (ρ +0.52, n 63).**

Rules of thumb: trust a receiver's 40 and tracked top speed; compare linemen only with linemen of their weight and role; treat defensive-back numbers as one input among several. Every line carries its n and interval.

In [ ]:
order = ["WR", "DL", "DB", "TE"]; names = {"WR": "Receivers", "DL": "Defensive line (all linemen)", "DB": "Defensive backs (CB + S)", "TE": "Tight ends (too few players)"}
cd_ = card[card.subgroup == "All"].copy(); cd_["o"] = cd_.group.map(order.index); cd_ = cd_.sort_values(["o"], kind="stable")
fig, ax = plt.subplots(figsize=(8.6, 9.6)); y = 0; yt, yl = [], []
for grp in order:
    d_ = cd_[cd_.group == grp]
    ax.text(-.78, y, names[grp], fontsize=10, fontweight="bold", va="center", transform=ax.get_yaxis_transform(), clip_on=False, ha="left"); y -= 1
    for _, r in d_.iterrows():
        col = VCOL[r.verdict]
        ax.axhline(y, color="#eeeeee", lw=.6, zorder=0)
        draw_ci(ax, y, r.ci_lo, r.ci_hi, r.n, col, 2.6); ax.scatter(r.rho_size, y, s=56, color=col, zorder=3); ax.scatter(r.rho_raw, y, s=36, facecolor="white", edgecolor=col, marker="D", zorder=3)
        yt.append(y); yl.append(f"{r.test_label.replace(' (tracking)', '')} → {r.outcome_label.replace(' (Sunday Speed)', '').replace(' (PFR)', '')}\n{tag(r.n, r.fdr_q)} · MDE ±{r.mde80:.2f}")
        y -= 1
    y -= .5
ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=8); ax.axvline(0, color="k", lw=.8); ax.set_xlim(-.9, .9); ax.set_ylim(y + .3, .8)
ax.set_xlabel("size-controlled rank correlation; + = better athlete, better result\nfilled dot = estimate, 95% interval; hollow = before size control")
fig.subplots_adjust(left=.43, right=.97, top=.90, bottom=.09)
fig.suptitle("Scorecard: one test per result, picked for the smallest p (so optimistic)", x=.01, y=.985, ha="left", va="top", fontweight="bold", fontsize=10.5)
fig.legend([ml.Line2D([], [], marker="o", color=VCOL[k], lw=0) for k in ["TRANSLATES", "LEANS", "SIZE, NOT SPEED", ND, "TOO FEW TO TELL"]], ["TRANSLATES", "LEANS", "SIZE, NOT SPEED", "NOT DETECTED (low power)", "TOO FEW TO TELL"], loc="upper left", bbox_to_anchor=(.01, .955), ncol=3, fontsize=8.5, frameon=False)
fig.savefig(SAVE / "fig7_scorecard.png"); plt.show()

*Figure 7. The scorecard, with the smallest detectable effect (MDE). Green translates, orange leans, purple mostly size, gray not detected.*

## Methods, in plain language
- **Size control.** Every link is repeated holding draft year, weight and height fixed (plus CB versus S, edge versus interior). Hollow marks are the link before size.
- **Shrinkage and held-out classes.** Rates from few snaps are pulled toward the group average. We train on two draft classes, predict the third, and ask whether adding the test beats a size-only model.
- **False-discovery control.** Each p comes from 10,000 permutations, each interval from a player-level bootstrap, and each result carries a Benjamini-Hochberg q across its whole family of tests. Verdict rules and receiver hypotheses were written first, and every p and interval was re-checked with a second method.
- **Power.** The smallest detectable effect (MDE, 80% power) is a Fisher-z approximation, checked by simulation (77% to 82% power at the MDE).

**Verdicts.** TRANSLATES: q below 0.05, interval excludes zero, held-out classes agree. LEANS: interval excludes zero or p below 0.05, but q does not clear. SIZE, NOT SPEED: a raw link of at least 0.30 that falls by half or more. TOO FEW: under 25 players. **NOT DETECTED (low power):** none of the above; no link above the MDE was seen, not no link.

## Limitations
- Rookies only (2023 to 2025 classes) with tracked snaps; skipped drills are not random.
- Counts are small: receivers about 60, defensive backs about 59, linemen about 70, tight ends about 21. Tight ends are all too few to tell (0.66 detectable, 20% power at 0.30).
- The same players produced the leads and test them; three held-out classes are the only out-of-sample check. Burst Score recipes were chosen using game results, and Figures 5 and 7 show the best-looking test per result, so those numbers are optimistic.
- Pooled defensive-line size may partly mean role; edge and interior splits have only 39 and 33 players (about 0.48 and 0.51 detectable).
- No blocker or quarterback tracks are linked: pressure and sack results are association, not causation.

## Appendix
Code cells are collapsed. The notebook reads the audited result tables from the `bdb2027-outputs` dataset and the raw competition files; attach both.